In [ ]:
import pandas as pd
import numpy as np

from src.utils import read_yml, Dict, str2time

In [ ]:
conf = Dict(read_yml("etc/thesis_config.yaml"))

In [ ]:
weather = pd.read_excel("data/processed_data/dvdk_weather.xlsx")
fm1 = pd.read_excel("data/processed_data/ok_1h.xlsx")
fm10 = pd.read_excel("data/processed_data/ok_10h.xlsx")
fm100 = pd.read_excel("data/processed_data/ok_100h.xlsx")
fm1000 = pd.read_excel("data/processed_data/ok_1000h.xlsx")

In [ ]:
wtrain = weather[(weather.utc >= conf.train_start) & (weather.utc <= conf.train_end)]
wval = weather[(weather.utc >= conf.val_start) & (weather.utc <= conf.val_end)]
wtest  = weather[(weather.utc >= conf.f_start) & (weather.utc <= conf.f_end)]

In [ ]:
# 1h
fm1_train = fm1[(fm1.utc_rounded >= conf.train_start) & (fm1.utc_rounded <= conf.train_end)]
fm1_val  = fm1[(fm1.utc_rounded >= conf.val_start) & (fm1.utc_rounded <= conf.val_end)]
fm1_test  = fm1[(fm1.utc_rounded >= conf.f_start) & (fm1.utc_rounded <= conf.f_end)]

# 10h
fm10_train = fm10[(fm10.utc_rounded >= conf.train_start) & (fm10.utc_rounded <= conf.train_end)]
fm10_val  = fm10[(fm10.utc_rounded >= conf.val_start) & (fm10.utc_rounded <= conf.val_end)]
fm10_test  = fm10[(fm10.utc_rounded >= conf.f_start) & (fm10.utc_rounded <= conf.f_end)]

# 100h
fm100_train = fm100[(fm100.utc_rounded >= conf.train_start) & (fm100.utc_rounded <= conf.train_end)]
fm100_val  = fm100[(fm100.utc_rounded >= conf.val_start) & (fm100.utc_rounded <= conf.val_end)]
fm100_test  = fm100[(fm100.utc_rounded >= conf.f_start) & (fm100.utc_rounded <= conf.f_end)]

# 1000h
fm1000_train = fm1000[(fm1000.utc_rounded >= conf.train_start) & (fm1000.utc_rounded <= conf.train_end)]
fm1000_val  = fm1000[(fm1000.utc_rounded >= conf.val_start) & (fm1000.utc_rounded <= conf.val_end)]
fm1000_test  = fm1000[(fm1000.utc_rounded >= conf.f_start) & (fm1000.utc_rounded <= conf.f_end)]

In [ ]:
weather.columns

In [ ]:
import pandas as pd

# Metadata dictionary
variable_info = {
    'Ed': {'units': '%', 'desc': 'Drying Equilibrium, derived from RH and temperature.'},
    'Ew': {'units': '%', 'desc': 'Wetting Equilibrium, derived from RH and temperature.'},
    'solar': {'units': 'W/m²', 'desc': 'Downward shortwave radiative flux.'},
    'wind': {'units': 'm/s', 'desc': 'Wind speed at 10m.'},
    'rain': {'units': 'mm/h', 'desc': 'Rain accumulation over the hour.'},
    'hod_utc': {'units': 'hours', 'desc': 'Hour of day, UTC time.'},
    'doy_utc': {'units': 'days', 'desc': 'Day of year, UTC time.'},
    'elevation': {'units': 'meters', 'desc': 'Height above sea level.'},
    'longitude': {'units': 'degree', 'desc': 'X-coordinate (longitude).'},
    'latitude': {'units': 'degree', 'desc': 'Y-coordinate (latitude).'},
}

# Columns from the DataFrame
cols_df = ['Ed', 'Ew', 'solar', 'wind', 'rain', 'hod_utc', 'doy_utc']

# Columns from conf object (single value)
cols_conf = {
    'elevation': 'ok_elev',
    'longitude': 'ok_lon',
    'latitude': 'ok_lat'
}

# Map column names to descriptive names
col_name_map = {
    'hod_utc': 'Hour of Day',
    'doy_utc': 'Day of Year',
}

summary_list = []

# Summarize DataFrame columns
for col in cols_df:
    var_name = col_name_map.get(col, col)
    mean_val = round(weather[col].mean(), 2)
    low = round(weather[col].min(), 2)
    high = round(weather[col].max(), 2)
    summary_list.append({
        'Variable': var_name.capitalize(),
        'Units': variable_info[col]['units'],
        'Description': variable_info[col]['desc'],
        'Mean': mean_val,
        '(Low, High)': f"({low}, {high})"  # <-- parentheses and comma
    })

# Summarize single-value columns from conf
for col, attr in cols_conf.items():
    val = round(getattr(conf, attr), 2)
    summary_list.append({
        'Variable': col.capitalize(),
        'Units': variable_info[col]['units'],
        'Description': variable_info[col]['desc'],
        'Mean': val,
        '(Low, High)': f"({val}, {val})"
    })

# Build summary DataFrame
summary_df = pd.DataFrame(summary_list)

# Reorder columns: Mean first
summary_df = summary_df[['Variable', 'Units', 'Description', 'Mean', '(Low, High)']]

summary_df

In [ ]:
# Map Fuel Class to (DataFrame, column name)
fm_dict = {
    'FM1': (fm1, 'fm1'),
    'FM10': (fm10, 'fm10'),
    'FM100': (fm100, 'fm100'),
    'FM1000': (fm1000, 'fm1000')
}

summary_list = []

for name, (df, col) in fm_dict.items():
    mean_val = round(df[col].mean(), 2)
    low = round(df[col].min(), 2)
    high = round(df[col].max(), 2)
    ncount=df.shape[0]
    summary_list.append({
        'Fuel Class': name,
        'N. Observations': ncount,
        'Mean': mean_val,
        '(Low, High)': f"({low}, {high})"
    })

# Build the summary DataFrame
fm_summary = pd.DataFrame(summary_list)

fm_summary

In [ ]:
# Map sets to their start/end dates
set_dates = {
    'Training': (conf.train_start, conf.train_end),
    'Validation': (conf.val_start, conf.val_end),
    'Test': (conf.f_start, conf.f_end)
}

# Organize weather splits
weather_splits = {
    'Training': wtrain,
    'Validation': wval,
    'Test': wtest
}

# Organize fuel moisture splits
fm_splits = {
    'FM1': (fm1_train, fm1_val, fm1_test),
    'FM10': (fm10_train, fm10_val, fm10_test),
    'FM100': (fm100_train, fm100_val, fm100_test),
    'FM1000': (fm1000_train, fm1000_val, fm1000_test)
}

# Build summary table
summary_list = []

for i, set_name in enumerate(['Training', 'Validation', 'Test']):
    start_date, end_date = set_dates[set_name]
    row = {
        'Set': set_name,
        'Start Date (UTC)': str2time(start_date).strftime('%Y-%m-%d, %H:%M'),
        'End Date (UTC)': str2time(end_date).strftime('%Y-%m-%d, %H:%M'),
        'Weather': len(weather_splits[set_name])
    }
    
    # FM row counts
    for fm_name, splits in fm_splits.items():
        row[fm_name] = len(splits[i])
    
    summary_list.append(row)

# Convert to DataFrame
split_summary = pd.DataFrame(summary_list)

split_summary

In [ ]:
transposed = split_summary.set_index('Set').T.reset_index()

# Rename the first column to 'Variable'
transposed = transposed.rename(columns={'index': 'Variable'})

transposed